# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library for reproducible scientific data analysis.

### Dataset Source
The dataset is defined by a Croissant schema, provided via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and tabular records using `mlcroissant`. The metadata provides schema-level detail, including available record sets and field definitions.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Metadata (schema) overview
metadata = dataset.metadata  # as single object, not subscripted
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Number of authors:", len(metadata.author) if hasattr(metadata, 'author') else 'N/A')
print("Published:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Let's review the available record sets, fields (columns), and their `@id`s from the schema. Referencing entities by their `@id` ensures clarity and reproducibility.

We will print information about record sets and their fields.

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata. Trying to access via .recordSets property...")
    record_sets = getattr(dataset.metadata, 'recordSets', [])

if record_sets:
    for rs in record_sets:
        print("\nRecord set @id:", rs['@id'])
        print("Name:", rs.get('name'))
        fields = rs.get('field', [])
        print("Available fields/columns:")
        for f in fields:
            print("  -", f.get('@id'), ":", f.get('name'))
else:
    print("Record sets are not explicitly listed in metadata. Listing available columns from extracted records.")
    # As a fallback, print the keys from the first record

## 3. Data Extraction
Load data from the record sets into pandas DataFrames. All entities are referenced by their `@id`. 

If the dataset provides a single record set or is flat, we extract all tabular rows available via the main record set.

In [ ]:
# Attempt to enumerate available record sets
record_sets_ids = []
record_sets = dataset.metadata.recordSet
if record_sets:
    record_sets_ids = [rs['@id'] for rs in record_sets]
else:
    # Fallback: Try to infer a main record set ID from dataset.records()
    all_records = list(dataset.records())
    if all_records:
        print("Sample record keys:", all_records[0].keys())
    # As Croissant expects recordSet specification, leave as empty list
    record_sets_ids = []

dataframes = {}
if record_sets_ids:
    for record_set_id in record_sets_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")
else:
    # If no recordSet information, extract all records as one DataFrame
    records = list(dataset.records())
    df = pd.DataFrame(records)
    main_record_set_id = 'main'  # Arbitrary ID as fallback
    dataframes[main_record_set_id] = df
    print(f"Loaded DataFrame with shape {df.shape}")

# Preview columns for the first DataFrame
selected_record_set_id = record_sets_ids[0] if record_sets_ids else main_record_set_id
print("Columns available:", dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now let's apply basic EDA: filtering, normalization, grouping by key attributes.

We use field `@id`s for all column references.

In [ ]:
# Select a numeric field for analysis
df = dataframes[selected_record_set_id]
# Infer candidate numeric fields by dtype
numeric_fields = list(df.select_dtypes(include=['int', 'float']).columns)
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = None
    print("No numeric fields found. Unable to proceed with numerical EDA.")

# If a numeric field exists, filter, normalize, and group
if numeric_field:
    # Set an arbitrary threshold for demonstration
    threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {{numeric_field}} > {{threshold}}:")
    print(filtered_df.head())

    # Normalize
    filtered_df["%s_normalized" % numeric_field] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {{numeric_field}} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized" ]].head())

    # Attempt to group by a categorical field
    categorical_fields = list(df.select_dtypes(include=['object', 'category']).columns)
    group_field = categorical_fields[0] if categorical_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {{group_field}}:")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Let's visualize the data distributions and relationships between fields. For this demonstration, we'll plot histograms of the selected numeric field and bar plots grouped by a categorical attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram of numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Barplot by group_field if present
    if group_field:
        plt.figure(figsize=(8,5))
        mean_values = df.groupby(group_field)[numeric_field].mean().dropna()
        mean_values.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we:

- Loaded FAIR^2 dataset metadata and records using the mlcroissant library.
- Reviewed dataset structure via record sets and field `@id`s, ensuring reproducible referencing.
- Demonstrated basic EDA: filtering, normalization, and grouping, visualized distributions and group means.
- All references to records, fields, and columns are shown via their `@id` per FAIR principles.

To extend this analysis, see the dataset documentation or the Croissant schema for additional contextual variables and advanced processing options.